# UG-AutoMedSAM2
### Inference and experimentation

This notebook provides a step-by-step guide to running and experimenting with UG-AutoMedSAM2, supporting future research and development.

> **Method and results:** See the report included in this repository for the method design and evaluation results.

## Workflow

1. **Data loading**
2. **Uncertainty envelope generation**
3. **Prompt generation**
4. **MedSAM2 inference**
5. **Evaluation**

---

## 0. Imports

In [1]:
import pandas as pd
from pathlib import Path

from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, evaluate_slice_by_slice
from plotting_functions import interactive_inference_viewer

## 1. Data loading

In [2]:
parentfolder = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"
subject_nr = 0
subject_name = sorted([p.name for p in Path(parentfolder).iterdir() if p.is_dir()])[subject_nr]
verbose = False
volume_of_interest = "CTVT"

DataLoader = DataLoader(parentfolder=parentfolder,subject_nr=subject_nr,volume_of_interest=volume_of_interest,verbose=verbose)

We also instantiate the unc_handler.

In [3]:
unc_handler = UG_prompter(data=DataLoader)

And seg_handler.

For using different valued mask prompts, mask bypass and occlusion logit prediction should be set to False. Since mask bypass completely skips the mask encoder and hardcodes whatever values you used to -10 and +10, and the occlusion logit is also quick to predict slices that went through the mask encoder as occluded, and hardcodes the output to -1024.

In [4]:
#Input variables for seg_handler, determine internal SAM processes/configs
used_checkpoint="checkpoints/MedSAM2_latest.pt" 
used_config="configs/sam2.1_hiera_t512.yaml"

#If mask bypass is True, it skips the internal mask encoder and sets the mask prompts to -10 and +10 logit scores.
allow_mask_bypass = True

#If use occlusion logit is set to true, it uses occlusion prediction within SAM2 and sets the value of mask occlusion predictions to -1024. 
use_occlusion_logit = True

seg_handler = Segmentation(data=DataLoader, checkpoint=used_checkpoint, cfg=used_config, use_mask_input_as_output_without_sam=allow_mask_bypass, pred_obj_scores=use_occlusion_logit)

c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


## 2. Uncertainty Envelope generation

In [ ]:
unc_map_bin = unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=3.0, mode="median")

array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, Fal

## 3. Prompt generation

In [6]:
staple_prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=4.0,interpix_dist=5,pixel_interval=10,) #Generates staple prompts.

bbox_and_mask = unc_handler.generate_prompts_boxes(band_threshold=3.0, interpix_dist=5, pixel_interval=10) #Generates bounding box prompts.

dense_prompt = seg_handler.load_dense_prompt() #Loads coarse mask prompt

staple_and_mask = combine_prompt_sets(prompt_dict_list = [dense_prompt, staple_prompts]) #combines staple prompts with coarse mask prompt

27 n_results: 8 max_total: 24.729200437664986 max_inner: 7.969600141048431 max_outer: 18.51760032773018
28 n_results: 8 max_total: 12.071600213646889 max_inner: 9.727600172162056 max_outer: 6.680400118231773
29 n_results: 8 max_total: 7.266400128602982 max_inner: 4.102000072598457 max_outer: 4.570800080895424
30 n_results: 8 max_total: 5.391200095415115 max_inner: 2.6956000477075577 max_outer: 3.1644000560045242
31 n_results: 8 max_total: 4.922400087118149 max_inner: 2.226800039410591 max_outer: 3.633200064301491
32 n_results: 8 max_total: 4.102000072598457 max_inner: 2.226800039410591 max_outer: 2.6956000477075577
33 n_results: 8 max_total: 3.9848000705242157 max_inner: 2.226800039410591 max_outer: 2.6956000477075577
34 n_results: 8 max_total: 3.516000062227249 max_inner: 2.226800039410591 max_outer: 2.6956000477075577
35 n_results: 8 max_total: 3.9848000705242157 max_inner: 2.226800039410591 max_outer: 2.8128000497817993
36 n_results: 8 max_total: 3.633200064301491 max_inner: 1.87520

## 4. MedSAM2 inference

First, prompt strategies are compiled into the segmentation module to allow organized processing.

In [7]:
prompt_sets = [staple_and_mask, bbox_and_mask]
prompt_names = ["Staple + Coarse", "Bbox + Coarse"]
prompt_strat_weights = [0.4 , 0.6]

#Compiles prompt strategies into one dictionary for segmentation handler to use.
seg_handler.compile_prompt_sets(prompt_dict_list=prompt_sets, prompt_set_names=prompt_names, prompt_set_weights=prompt_strat_weights)

Then, we run inference using the configured prompt strategies.

In [8]:
pred_seg = seg_handler.run_segmentation_sets(propagation_style='default', #Multiple propagation styles are supported. Relevance to performance was limited though.
                                  weighting_strategy="custom", #If set to average, does not use custom weights.
                                  threshold=0.0, #Threshold for logit scores to be considered as foreground. 
                                  no_external_propagation=True) #Limits propagation to the first and last slice that contained coarse mask.

Running segmentation for prompt set 'Staple + Coarse' with slices: [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:1157: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Forward propagation (default, from slice 27)...


propagate in video: 100%|██████████| 18/18 [00:00<00:00, 67.22it/s]


Backward propagation (default, from slice 44)...


propagate in video: 100%|██████████| 18/18 [00:00<00:00, 185.28it/s]


Running segmentation for prompt set 'Bbox + Coarse' with slices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Forward propagation (default, from slice 27)...


propagate in video: 100%|██████████| 18/18 [00:00<00:00, 63.52it/s]


Backward propagation (default, from slice 44)...


propagate in video: 100%|██████████| 18/18 [00:00<00:00, 217.01it/s]


Lastly, we perform post-processing if applicable/relevant.

In [9]:
seg = seg_handler.remove_distant_slices(tolerance_frames=0) #Remove slice content that is too far away from original coarse mask. Pointless if no_external_propagation is set to True.

## 5. Evaluation

First we load the recontour consensus to evaluate the created segmentation against.

In [10]:
recontours = DataLoader.load_recontours() #Loads observer recontours 
consensus = DataLoader.load_consensus(indices = [0, 1, 2, 3])

In [11]:
volume_evaluator = Evaluator(
    pred=seg_handler.predicted_seg,
    gt=DataLoader.consensus_recontour,
    spacing=DataLoader.img_spacing,
    subject_name=subject_name
)
volume_metrics = volume_evaluator.compute_all(surface_dice_tol=1.0)
volume_df = pd.DataFrame([volume_metrics])


slice_results = evaluate_slice_by_slice(
    pred=seg_handler.predicted_seg,
    gt=DataLoader.consensus_recontour,
    uncertainty=DataLoader.unc_map,
    spacing=DataLoader.img_spacing,
    surface_dice_tol=1.0,
    subject_name=subject_name
)

slice_df = pd.DataFrame(slice_results)

Evaluating slices 27 to 44...
Number of evaluated slices: 18


In [12]:
print("Volume-level metrics")
display(volume_df.round(3))

print("Slice-level metrics")
display(slice_df.round(3))

Volume-level metrics


,subject name,HD_mm,HD95_mm,MSD_mm,ASSD_mm,Dice,SurfaceDice@1.0mm,SurfaceDice@2.0mm,CentroidDistance_mm,PredictionVolume_mm3,GroundTruthVolume_mm3,AbsVolumeDifference_mm3,RelativeVolumeDifference_percent
0,newAcq_050f229dc2bdb64c,4.745,1.482,0.299,0.284,0.969,0.907,0.972,0.258,52013.232,51386.878,626.354,1.219


Slice-level metrics


,subject,slice_idx,HD_mm,HD95_mm,MSD_mm,ASSD_mm,Dice,SurfaceDice@1.0mm,SurfaceDice@2.0mm,CentroidDistance_mm,...,HasGroundTruth,HasUncertainty,PredictionWithoutGroundTruth,GroundTruthWithoutPrediction,BothEmpty,BothPresent,Oversegmentation,Undersegmentation,error,relative slice idx
0,newAcq_050f229dc2bdb64c,27,9.665,7.329,3.793,3.043,0.513,0.102,0.280,5.185,...,True,True,False,False,False,True,True,False,NaN,0.000
1,newAcq_050f229dc2bdb64c,28,2.813,2.344,1.264,1.199,0.859,0.591,0.831,1.712,...,True,True,False,False,False,True,True,False,NaN,0.059
2,newAcq_050f229dc2bdb64c,29,4.617,3.421,1.017,0.968,0.914,0.676,0.903,1.237,...,True,True,False,False,False,True,True,False,NaN,0.118
3,newAcq_050f229dc2bdb64c,30,1.690,1.406,0.554,0.549,0.962,0.860,1.000,0.190,...,True,True,False,False,False,True,True,False,NaN,0.176
4,newAcq_050f229dc2bdb64c,31,2.852,2.734,0.971,0.974,0.940,0.644,0.849,1.008,...,True,True,False,False,False,True,False,True,NaN,0.235
5,newAcq_050f229dc2bdb64c,32,1.482,1.048,0.516,0.528,0.971,0.884,1.000,0.531,...,True,True,False,False,False,True,False,True,NaN,0.294
6,newAcq_050f229dc2bdb64c,33,0.663,0.469,0.256,0.256,0.989,1.000,1.000,0.065,...,True,True,False,False,False,True,True,False,NaN,0.353
7,newAcq_050f229dc2bdb64c,34,0.938,0.938,0.248,0.247,0.989,1.000,1.000,0.114,...,True,True,False,False,False,True,True,False,NaN,0.412
8,newAcq_050f229dc2bdb64c,35,0.938,0.469,0.236,0.235,0.990,1.000,1.000,0.040,...,True,True,False,False,False,True,True,False,NaN,0.471
9,newAcq_050f229dc2bdb64c,36,0.938,0.938,0.308,0.306,0.988,1.000,1.000,0.041,...,True,True,False,False,False,True,True,False,NaN,0.529


In [ ]:
viewer = interactive_inference_viewer(
    data=DataLoader,
    seg_handler=seg_handler,
    prompter=unc_handler,
    title=str(subject_name),
)